# 02 - Reproduce a program-space benchmark number

`core/benchmark/program_space_primary.csv` reports, per context, the mean
per-program Pearson correlation of the reference model's predicted vs
measured usage coordinates (`prog_pearson_mean`) against a permutation
null (`null_mean`, `null_sd`), with the resulting `z`. The model
predictions themselves are not in `core/`, but the **null scale** is
recomputable from the shipped usages alone, and the fold convention is
verifiable exactly. This notebook does both.

**Approximation, stated up front:** the published nulls were computed per
evaluation cell (held-out compounds of each fold); here we shuffle the
compound pairing over one context's full measured usage matrix. Both
estimate the sampling distribution of per-program Pearson under no
association, so the recomputed null mean/sd should land close to the
published values. Treat this as a cross-check of scale, not a bit-exact
reproduction.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Locate the package root (works whether the notebook runs from examples/
# or from the package root).
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "core" / "basis" / "basis_registry.json").exists())
sys.path.insert(0, str(ROOT / "src"))
print("package root located (all paths below are relative to it)")


package root located (all paths below are relative to it)


## Verify the fold convention

fold = SHA256(`public_compound_id`) mod 5 (full 64-hex digest as an
integer). `core/splits/fold_assignments.parquet` should match this rule
on every row, and fold 0 is the held-out test bed (the shared basis and
the reference model were both fit without it).

In [2]:
import hashlib

folds = pd.read_parquet(ROOT / "core" / "splits" / "fold_assignments.parquet")
print("rows:", len(folds), "| fold counts:", folds["fold"].value_counts().sort_index().to_dict())

def sha_fold(compound_id: str) -> int:
    return int(hashlib.sha256(compound_id.encode()).hexdigest(), 16) % 5

sample = folds.sample(n=20000, random_state=0)
recomputed = sample["public_compound_id"].map(sha_fold)
print("fold rule matches on",
      int((recomputed == sample["fold"]).sum()), "/", len(sample), "sampled rows")


rows: 190699 | fold counts: {0: 38652, 1: 38362, 2: 37801, 3: 38106, 4: 37778}
fold rule matches on 20000 / 20000 sampled rows


## Recompute the permutation null from the usages

The published statistic is the mean per-program Pearson on held-out
compounds, and its null is a per-program pairing permutation. Rebuild it:
take a context's fold-0 compounds (the held-out test fold), shuffle the
compound pairing, compute the per-program Pearson for each of the 32
programs, and pool the draws. The pooled null mean/sd should reproduce
the published `null_mean` / `null_sd`, and re-plugging the published
`prog_pearson_mean` into the recomputed null should reproduce `z`.

**Approximation, stated up front:** the published nulls average over the
evaluation's fold x seed cells; here we use each context's fold-0 cell
(same construction, one cell), so expect agreement to about three
decimals, not bitwise identity.

In [3]:
def usage_pairing_null(u: np.ndarray, n_draws: int = 200, seed: int = 0):
    """Per-program pairing-permutation null: shuffle compound pairing,
    compute per-program Pearson, pool all draws across programs."""
    rng = np.random.default_rng(seed)
    u = u.astype(np.float64)
    pooled = []
    for _ in range(n_draws):
        perm = rng.permutation(len(u))
        for j in range(u.shape[1]):
            pooled.append(np.corrcoef(u[:, j], u[perm, j])[0, 1])
    return float(np.mean(pooled)), float(np.std(pooled))

folds = pd.read_parquet(ROOT / "core" / "splits" / "fold_assignments.parquet")
ref = pd.read_csv(ROOT / "core" / "benchmark" / "program_space_primary.csv")
rows = []
for ctx in ["zel039_aec7", "zel024_hek293"]:
    u = np.load(ROOT / "core" / "usages" / f"usages_{ctx}.npy")
    compounds = pd.read_parquet(ROOT / "core" / "usages" / f"usages_{ctx}_compounds.parquet")
    fold0_ids = set(folds.query("context == @ctx and fold == 0")["public_compound_id"])
    u0 = u[compounds["public_compound_id"].isin(fold0_ids).to_numpy()]
    null_mean, null_sd = usage_pairing_null(u0)
    published = ref.query(
        "arm == 'context_token_trunk' and k == 32 and "
        "split_type == 'compound_5fold' and context == @ctx").iloc[0]
    z_recomputed = (published["prog_pearson_mean"] - null_mean) / null_sd
    rows.append({
        "context": ctx,
        "n_fold0": len(u0),
        "published_null_mean": round(published["null_mean"], 5),
        "recomputed_null_mean": round(null_mean, 5),
        "published_null_sd": round(published["null_sd"], 5),
        "recomputed_null_sd": round(null_sd, 5),
        "prog_pearson_mean": round(published["prog_pearson_mean"], 4),
        "published_z": round(published["z"], 1),
        "z_with_recomputed_null": round(z_recomputed, 1),
    })
print(pd.DataFrame(rows).to_string(index=False))


      context  n_fold0  published_null_mean  recomputed_null_mean  published_null_sd  recomputed_null_sd  prog_pearson_mean  published_z  z_with_recomputed_null
  zel039_aec7     4215              0.00001              -0.00047            0.01545             0.01551             0.3206         20.8                    20.7
zel024_hek293     2811             -0.00023              -0.00070            0.01890             0.01885             0.5302         28.1                    28.2


## Reading the result

The recomputed null reproduces the published `null_mean` and `null_sd`
to about three decimals, and the z-scores match to within rounding: the
reference model's program-space signal is 20-28 null standard deviations
above chance in these two contexts. The exact reference numbers to quote
remain the published ones in `core/benchmark/program_space_primary.csv`;
the reference scores for the zel031_a549 exception live in
`core/benchmark/per_context_comparison_k32.csv`.